# R-Build v2 — interactive demo (Colab / Jupyter)
Cache-loop → parallel-bundle architecture, every value user-modifiable.

In [ ]:
# install (in Colab: point at your copy of the repo/wheel)
!pip install -q .[notebook]  # or: pip install rbuild-engine-2.0.0-py3-none-any.whl[notebook]

In [ ]:
from rbuild import RBuildConfig, RBuildModel, Trainer, preset, interactive
import torch

## 1. Interactive panel — tweak every value, watch the live report, click Build

In [ ]:
ui = interactive.launch(preset('s1'))   # or RBuildConfig() for a blank tiny config

## 2. Or modify everything in code

In [ ]:
cfg = preset('s1')
cfg.cache_loop.n_loops = 8        # how often the line loops to pull cache
cfg.parallel.n_branches = 8       # parallel branches per stage
cfg.parallel.bundle_mode = 'gate' # gate | mean | concat
cfg.train.precision = 'bf16'
print(cfg.report())               # params + naive-vs-optimized cost

## 3. Build + train (toy random-data smoke test)

In [ ]:
cfg.model.vocab_size = 8000   # shrink for the demo
model = RBuildModel(cfg)
assert sum(p.numel() for p in model.parameters()) == cfg.count_parameters()['total_params']

def batches():
    while True:
        yield torch.randint(0, 8000, (4, 64)), torch.randint(0, 8000, (4, 64))

trainer = Trainer(model, cfg)
trainer.fit(batches(), max_steps=50)

## 4. Gradient-free fact write + generation

In [ ]:
model.remember(torch.randint(0, 8000, (16,)))   # write to fast-weight cache, no training
out = model.generate(torch.randint(0, 8000, (1, 8)), max_new_tokens=32, temperature=0.8, top_p=0.9)
out

## 5. Interactive session — /remember, /temp, /topp, /config, /forget

In [ ]:
vocab = 8000
interactive.chat(model,
                 encode=lambda s: torch.randint(0, vocab, (max(1, len(s)//2),)),  # swap in your tokenizer
                 decode=lambda ids: ' '.join(map(str, ids.tolist())))

## 6. Checkpoint — memory matrix rides along

In [ ]:
trainer.save_checkpoint('ckpt_s1')
model2 = Trainer.load_checkpoint('ckpt_s1')
torch.equal(model.memory.matrix, model2.memory.matrix)